In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    pa.campaign_id
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id
WHERE o.order_status IN ('Completed', 'Shipped')
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

,order_id,customer_id,order_date,order_status,total_price_before_tax,product_id,quantity,product_name,unit_price,line_price_before_tax,is_free_gift,campaign_id
0,37,157800,2023-03-25 11:47:39,Completed,317.79,105,1,Ray-Ban Flacko Prescription,188.26,150.61,False,1
1,37,157800,2023-03-25 11:47:39,Completed,317.79,97,1,Ray-Ban New Wayfarer Prescription,208.97,167.18,False,1
2,95,132132,2023-01-07 01:43:38,Completed,284.77,16,1,Ray-Ban Flacko Prescription,355.96,284.77,False,1
3,109,76594,2023-02-22 10:47:17,Completed,1070.81,23,3,Ray-Ban Sam Prescription,435.29,1070.81,False,3
4,133,81844,2023-02-09 00:08:23,Shipped,609.49,14,1,Ray-Ban Ray-Ban Reverse Prescription,281.05,224.84,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...
229638,499963,72391,2024-11-25 16:09:59,Shipped,615.53,42,1,Ray-Ban Ray-Ban Reverse Non-prescription,411.78,267.66,False,26
229639,499987,136134,2024-10-25 05:50:04,Completed,744.64,152,1,Ray-Ban Ray-Ban Meta Prescription,254.33,206.01,False,22
229640,499987,136134,2024-10-25 05:50:04,Completed,744.64,40,1,Ray-Ban Ray-Ban Meta Prescription,237.84,192.65,False,22
229641,499987,136134,2024-10-25 05:50:04,Completed,744.64,69,1,Ray-Ban Round Metal Non-prescription,427.14,345.98,False,22


In [8]:
df = df_order_completed_shipped.copy()
df['order_date'] = pd.to_datetime(df['order_date'])

# 年 / 季度 / 月 的字段
df['year'] = df['order_date'].dt.year

# 去重客户计数
year_counts = df.groupby('year')['customer_id'].nunique().reset_index(name='customer_count')

year_counts.to_parquet('2023-2024_each_year_customer_count.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_each_year_customer_count.parquet 文件")

year_counts

数据已保存为 2023-2024_each_year_customer_count.parquet 文件


,year,customer_count
0,2023,60470
1,2024,59240


In [ ]:
# 年 / 季度 / 月 的字段
df['quarter'] = df['order_date'].dt.to_period('Q').astype(str).str.replace(r'(\d{4})Q(\d)', r'\1-Q\2', regex=True)
# 去重客户计数
quarter_counts = df.groupby('quarter')['customer_id'].nunique().reset_index(name='customer_count')

quarter_counts.to_parquet('2023-2024_each_quarter_customer_count.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_each_quarter_customer_count.parquet 文件")

quarter_counts

数据已保存为 2023-2024_each_quarter_customer_count.parquet 文件


,quarter,customer_count
0,2023-Q1,11384
1,2023-Q2,10725
2,2023-Q3,16626
3,2023-Q4,32126
4,2024-Q1,11296
5,2024-Q2,17977
6,2024-Q3,14015
7,2024-Q4,26098


In [10]:
# 年 / 季度 / 月 的字段
df['month'] = df['order_date'].dt.to_period('M').astype(str)

# 去重客户计数
month_counts = df.groupby('month')['customer_id'].nunique().reset_index(name='customer_count')

month_counts.to_parquet('2023-2024_each_month_customer_count.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_each_month_customer_count.parquet 文件")

month_counts

数据已保存为 2023-2024_each_month_customer_count.parquet 文件


,month,customer_count
0,2023-01,4258
1,2023-02,3735
2,2023-03,3688
3,2023-04,3593
4,2023-05,3858
5,2023-06,3511
6,2023-07,5910
7,2023-08,5863
8,2023-09,5493
9,2023-10,11892
